In [1]:
from pyiceberg.catalog import load_catalog
import duckdb

PROJECT = "/Users/Shared/academy/learner/projects/ethereum_iceberg"
catalog = load_catalog(
    "local",
    type="sql",
    uri=f"sqlite:///{PROJECT}/catalog/pyiceberg_catalog.db",
    warehouse=f"file://{PROJECT}/warehouse",
)
S3 = "s3://aws-public-blockchain/v1.0/eth"
con = duckdb.connect()
con.execute("set s3_region = 'us-east-2'")
catalog.list_namespaces()

[('ref',), ('raw',)]

In [2]:
blocks = catalog.load_table("raw.blocks")
blocks.inspect.snapshots().select(["committed_at", "snapshot_id", "operation"]).to_pandas()

,committed_at,snapshot_id,operation
0,2026-11-13 15:16:34.881,8239508963356461323,append
1,2026-11-13 15:16:46.771,1664010884662275675,append
2,2026-11-13 15:16:49.580,7852830793590955055,append
3,2026-11-13 15:16:52.097,7488311288153616006,append
4,2026-11-13 15:16:53.971,7681222492598437330,append
5,2026-11-13 15:16:59.817,4039380342251045399,overwrite
6,2026-11-13 15:16:59.932,5865045181067126141,delete
7,2026-11-13 15:17:00.047,7328567001918218873,append
8,2026-11-13 15:17:04.100,6124517585318072545,append
9,2026-11-13 15:17:56.880,1696631914364866967,append


In [3]:
blocks.inspect.refs().to_pandas()

,name,type,snapshot_id,max_reference_age_in_ms,min_snapshots_to_keep,max_snapshot_age_in_ms
0,main,BRANCH,1696631914364866967,NaN,NaN,NaN
1,after-cleanup,TAG,1696631914364866967,NaN,NaN,NaN


In [4]:
blocks.inspect.manifests().select(["path", "added_data_files_count", "existing_data_files_count", "deleted_data_files_count"]).to_pandas()

,path,added_data_files_count,existing_data_files_count,deleted_data_files_count
0,file:///Users/Shared/academy/learner/projects/...,1,0,0
1,file:///Users/Shared/academy/learner/projects/...,1,0,0
2,file:///Users/Shared/academy/learner/projects/...,1,0,0
3,file:///Users/Shared/academy/learner/projects/...,4,0,0


In [5]:
blocks.inspect.files().select(["file_path", "record_count", "file_size_in_bytes", "spec_id"]).to_pandas()

,file_path,record_count,file_size_in_bytes,spec_id
0,file:///Users/Shared/academy/learner/projects/...,7182,1457746,1
1,file:///Users/Shared/academy/learner/projects/...,7176,1460076,0
2,file:///Users/Shared/academy/learner/projects/...,7176,1454923,0
3,file:///Users/Shared/academy/learner/projects/...,222424,43651510,0
4,file:///Users/Shared/academy/learner/projects/...,214993,42267230,0
5,file:///Users/Shared/academy/learner/projects/...,215179,42340420,0
6,file:///Users/Shared/academy/learner/projects/...,7173,1460850,0


In [6]:
transactions = catalog.load_table("raw.transactions")
transactions.inspect.partitions().to_pandas()

,partition,spec_id,record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
0,{'block_day': 2026-10-02},1,1875646,8,128337274,0,0,0,0,2026-11-13 15:18:12.582,7424009227275671105
1,{'block_day': 2026-10-01},1,2036041,9,143062624,0,0,0,0,2026-11-13 15:18:07.498,7857080667661415133
2,{'block_day': 2026-10-07},1,1963594,1,137620400,0,0,0,0,2026-11-13 15:17:56.579,5925522578341126574
3,{'block_day': 2026-10-06},1,1957602,1,134531223,0,0,0,0,2026-11-13 15:17:50.299,6399773433597975194
4,{'block_day': 2026-10-05},1,1774270,1,147848863,0,0,0,0,2026-11-13 15:17:42.920,7019737022404650187
5,{'block_day': 2026-10-04},1,1526358,1,108469864,0,0,0,0,2026-11-13 15:17:36.900,2539056203822355527
6,{'block_day': 2026-10-03},1,1562292,1,129557069,0,0,0,0,2026-11-13 15:17:30.582,7745473855276163365


In [7]:
blocks.inspect.entries().select(["status", "snapshot_id", "sequence_number"]).to_pandas().value_counts("status")

status
1    7
Name: count, dtype: int64